In [28]:
# STEP 1 - import + ROOT
from pathlib import Path
import geopandas as gpd

ROOT = Path.cwd()
OUT = ROOT / "geojson_simplified"
OUT.mkdir(exist_ok=True)

# STEP 2 - utility: pulizia + semplificazione robusta
def simplify_gdf(gdf: gpd.GeoDataFrame, tolerance_m: float) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)

    gdf_3857 = gdf.to_crs(epsg=3857)

    gdf_3857["geometry"] = gdf_3857["geometry"].buffer(0)

    gdf_3857["geometry"] = gdf_3857["geometry"].simplify(
        tolerance=tolerance_m,
        preserve_topology=True
    )

    gdf_out = gdf_3857.to_crs(epsg=4326)

    return gdf_out

# STEP 3 - set tolleranze (metri) per layer “pesanti”
TOLERANCE_BY_NAME = {
    "comuni": 120,
    "regioni": 300,
    "confini": 120,
    "uso_suolo": 250,
    "copertura": 250,
    "euap": 200,
    "aree_protette": 200,
    "aree_idonee": 200,
}
DEFAULT_TOL_M = 80

# STEP 4 - loop su tutti gli shapefile e export GeoJSON semplificati
shps = sorted(ROOT.glob("*.shp"))
print(f"Trovati {len(shps)} shapefile in: {ROOT}")

for shp in shps:
    name = shp.stem
    tol = DEFAULT_TOL_M
    name_lower = name.lower()

    for k, v in TOLERANCE_BY_NAME.items():
        if k in name_lower:
            tol = v
            break

    try:
        gdf = gpd.read_file(shp)
        gdf = simplify_gdf(gdf, tolerance_m=tol)

        gdf = gdf.loc[~gdf.geometry.is_empty & gdf.geometry.notna()].copy()

        out_path = OUT / f"{name}_wgs84_simplified.geojson"
        gdf.to_file(out_path, driver="GeoJSON")
        print(f"OK -> {out_path.name} | tol={tol}m | features={len(gdf)}")

    except Exception as e:
        print(f"ERRORE su {shp.name}: {e}")

# STEP 5 - (opzionale) suggerimento pratico finale
print("\nCarica su Kepler prima i layer più importanti e aggiungi gli altri uno per volta.")


Trovati 14 shapefile in: C:\Users\Stefa\Analisi Energetica Italiana
OK -> bi_energia_gwh_2023_wgs84_simplified.geojson | tol=80m | features=20
OK -> cab_prim_inv_flusso_2020_wgs84_simplified.geojson | tol=80m | features=7904
OK -> classificazione_sismica_2024_wgs84_simplified.geojson | tol=80m | features=7904
OK -> comuni_2025_wgs84_simplified.geojson | tol=120m | features=7896
OK -> consumi_autosufficienza_2015_wgs84_simplified.geojson | tol=80m | features=7868
OK -> consumi_totale_mwh_2015_wgs84_simplified.geojson | tol=80m | features=8046
OK -> eo_pot_comunale_2019_wgs84_simplified.geojson | tol=80m | features=7926
OK -> eolico_onshore_2023_wgs84_simplified.geojson | tol=80m | features=125
OK -> euap_2011_wgs84_simplified.geojson | tol=200m | features=876
OK -> fv_impianti_comune_2021_wgs84_simplified.geojson | tol=80m | features=7903
OK -> localita_2021_istat_2021_wgs84_simplified.geojson | tol=80m | features=69003
OK -> po_poten_comunali_2021_wgs84_simplified.geojson | tol=80m | f